```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

In [ ]:
%matplotlib inline

# 02.3 Inlet Profiles

## Contents

**Tools:** CADET-GUI widgets: model editors and event timeline; CADET-Process (Python).

**Python used:** Event calls, component lists, polynomial coefficients and plotting.

**Introduces:** events, polynomial inlet profiles, `plot_events()` and profiles from arrays.

1. Inlet-to-outlet model
2. Load, wash and elution
3. The profile in Python
4. Additional example: inlet profiles from data

```{note}
In the model templates, the LWE template created the events for us.
In this lesson, we build the inlet profile of an LWE ourselves, step by step, on the simplest system: an inlet and an outlet.
```

## The experiment: inlet to outlet

We define a **load, wash, elution and final wash** method for **Salt** and **Protein**, with a linear salt gradient during elution. The flow rate stays at **1 mL/s**.

An [Inlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html) connected directly to an [Outlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Outlet.html) lets us focus on the inlet profile.

```{figure} ../resources/first_steps/IO.png
:width: 30%
```

In [ ]:
# Load the shared GUI setup functions.
%run ../resources/first_steps/lesson_setup.py
# Create empty components, units and flow sheet for the widgets below.
component_system, units, flow_sheet, _ = empty_model("inlet_profile")

## Setting up the model

We enter `Salt` first and `Protein` second in **Components**. This order also determines the order of concentration values in the events. We then create an inlet and outlet and connect them.
**Show code** in each widget gives the CADET-Process code.

In [ ]:
from cadetgui import ComponentSystemWidget

ComponentSystemWidget(component_system).display()

### Inlet and outlet

The `Inlet` supplies liquid at concentration `c` and flow rate `flow_rate` (see the [Inlet reference](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html)).
Events can change either parameter during the process.
Here, the flow rate is fixed at $1~mL \cdot s^{-1}$ ($10^{-6}~m^3 \cdot s^{-1}$); the concentration is set by events.

The `Outlet` collects whatever reaches it.

In the unit editor, we create an `Inlet` named `inlet` and an `Outlet` named `outlet`. Under **Parameters of** `inlet`, `flow_rate` is set to `1e-6` m³/s and both constant concentrations to zero. Events will define the feed; the outlet needs no settings.

In [ ]:
from cadetgui import UnitOperationsWidget

UnitOperationsWidget(component_system, units).display()

### Flow sheet

Both units are added with **Unit to add** / **Add to flow sheet** and connected **From** `inlet` **To** `outlet` with **Connect**. Without a column, the outlet will show the imposed inlet profile:

In [ ]:
from cadetgui import FlowSheetWidget

FlowSheetWidget(flow_sheet, units=units).display()

In [ ]:
from cadetgui import ProcessWidget

# Create a 180 s process for the load, wash and elution events.
lwe_gui_process = lwe_gui_model(flow_sheet)

## Part 1: load, wash and elution

The inlet profile follows a typical method: **load** with **50 mM salt** and **1 mM protein** for **60 s**. Then a **wash** with **50 mM salt** for **30 s** stops the protein feed (**0 mM protein**). During **elution**, salt increases linearly from **50 to 500 mM** over **60 s**, still with **0 mM protein**. A **final wash** for **30 s** at **500 mM salt**, without protein, ends the method.

The `cycle_time` is **180 s**. Each change starts an event; the following steps translate this method into event times and concentration values. All event times lie within the cycle time: later times would be wrapped into the cycle ([99.6 Common Problems](../../99%20Misc/99.6%20Common%20Problems/)).

### Load and wash

The first two events set Salt and Protein to the constant values below; their slope and higher coefficients stay zero. Concentrations are in mM (equivalent to mol/m³).

| Event | Time / s | Salt | Protein |
|---|---:|---:|---:|
| load | 0 | 50 | 1 |
| wash | 60 | 50 | 0 |

In the timeline, the protein concentration drops to zero when washing starts, while salt stays at 50 mM.

### Elution gradient and final wash

During elution, raise salt from 50 to 500 mM over 60 s. The slope is $(500-50)/60 = 7.5$ mM/s.

`elute` starts at 90 s. In its polynomial editor, Salt gets the start value $a_0 = 50$ and the slope $a_1 = 7.5$; $a_2$, $a_3$ and Protein stay zero. `final_wash` at 150 s holds Salt constant at 500 with Protein zero, so the salt stops rising.

| Event | Time / s | Salt | Protein |
|---|---:|---|---:|
| elute | 90 | start 50, slope 7.5 | 0 |
| final_wash | 150 | constant 500 | 0 |

```{note}
The polynomial uses time since its event: $c(t)=a_0+a_1(t-t_e)+a_2(t-t_e)^2+a_3(t-t_e)^3$. The next event starts a new polynomial.
```

In the complete timeline, the salt rises during elution and stays at 500 mM during the final wash.

In the process editor, each event gets a name, a time, the parameter `flow_sheet.inlet.c` and its coefficients. The timeline shows the concentration profile as the events are added.

In [ ]:
ProcessWidget(lwe_gui_process).display()

## Part 2: the profile in Python

**Show code** in the process editor gives the process and its events as code. The same profile can be built directly in Python: the inlet-to-outlet model and the 180 s process are created as before, and each event is added with `add_event(name, parameter_path, state, time)`.

In [ ]:
from CADETProcess.processModel import ComponentSystem, Inlet, Outlet, FlowSheet, Process

component_system = ComponentSystem(["Salt", "Protein"])
inlet = Inlet(component_system, "inlet")
inlet.flow_rate = 1e-6  # 1 mL/s
inlet.c = [0, 0]
outlet = Outlet(component_system, "outlet")
flow_sheet = FlowSheet(component_system, "flow_sheet")
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(outlet)
flow_sheet.add_connection(inlet, outlet)

process = Process(flow_sheet, "lwe")
process.cycle_time = 180

All four events change `flow_sheet.inlet.c`. Their state lists one entry per component, Salt first.

| Event | Time / s | State |
|---|---:|---|
| `load` | 0 | `[50, 1]` |
| `wash` | 60 | `[50, 0]` |
| `elute` | 90 | `[[50, 7.5], [0]]` |
| `final_wash` | 150 | `[500, 0]` |

In [ ]:
# Add the four events from the table


```{note}
**One number or a list of coefficients per component**

Each concentration is a cubic polynomial with the coefficients $a_0, a_1, a_2, a_3$. The state can give each component a single number or a list of coefficients:

| State | Entry per component | Complete form |
|---|---|---|
| `[50, 0]` | a number: the constant $a_0$ | `[[50, 0, 0, 0], [0, 0, 0, 0]]` |
| `[[50, 7.5], [0]]` | a list $[a_0, a_1, \dots]$ | `[[50, 7.5, 0, 0], [0, 0, 0, 0]]` |

Missing coefficients are zero, so a constant collapses to a single number. As soon as one component needs a slope, every component gets its own list. A flat `[50, 7.5]` is **not** a gradient: it means 50 mM Salt and 7.5 mM Protein, both constant.
```

### Plot the profile

`plot_events()` shows the inlet profile directly from the events. Its curve matches the editor timeline.

In [ ]:
# Plot the concentration profiles defined by the events.
_ = process.plot_events()

The LWE template automates this recipe on an `LCFlowSheet`, mixing buffer A and B by their flow rates. Here we set inlet concentrations directly.

```{note}
Events define the inlet profile. The timeline and `plot_events()` show it without a transport simulation.
```

## Additional example: inlet profiles from data

For a measured or irregular inlet profile, `add_concentration_profile()` accepts arrays of times and concentrations. This example uses a synthetic profile with noise:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def noisy_inlet(t, seed=0):
    r = np.random.default_rng(seed)
    s = np.exp(-0.5*((t-30)/5)**2) + 0.7*np.exp(-0.5*((t-80)/9)**2)
    d = 0.03*np.sin(2*np.pi*0.02*t)
    n = r.normal(0, 0.02, t.size)*10
    k = np.exp(-np.abs(np.arange(-15, 16))/5)
    y = np.convolve(n, k, 'same')/k.sum()
    return np.clip(s + d + y, 0, None)

t = np.linspace(0, 120, 100)
c = noisy_inlet(t).reshape(-1, 1)

plt.plot(t, c)
plt.xlabel("Time / s")
plt.ylabel("Protein concentration / mol/m³")

For this example, salt stays at zero; only protein follows the supplied data. A fresh inlet-to-outlet model and a 120 s process, `profile_process`, keep this profile separate from the LWE example.

In [ ]:
# Create a fresh inlet-to-outlet model with zero salt and protein concentrations,
# and a process covering the 120 s of data.
profile_component_system, profile_units, profile_sheet, _ = empty_model("profile")
populate_inlet_model(profile_component_system, profile_units, profile_sheet)
profile_process = Process(profile_sheet, "profile")
profile_process.cycle_time = 120

`profile_process.add_concentration_profile()` turns the data into inlet events. It takes the inlet name `"inlet"`, the time array `t`, the concentration array `c` and the component the data belong to, `components=["Protein"]`.

In [ ]:
# Add the concentration profile for Protein


In [ ]:
# Plot the data-derived profile on the same time scale as the input data.
_ = profile_process.plot_events(x_axis_in_minutes=False)

## Summary

- `add_event()` defines when an inlet setting changes.
- Polynomial coefficients describe the concentration from each event onward.
- The editor timeline and `plot_events()` show the profile without a transport simulation.

**Now:** **02.3 Inlet Profiles: Exercises**: build a load–wash–elute profile and shorten its gradient.

**Next lesson:** **02.4 Residence Time Distributions** shows how mixer and tubing transport change a pulse.